# Optimizers in Deep Learning

**Learning Objectives:** Understand SGD, momentum, AdaGrad, RMSProp, Adam, AdamW, learning rate schedules. Compare optimizer performance and choose the right optimizer for different tasks.

## 1. Gradient Descent Optimization

The optimizer updates parameters to minimize the loss function. Vanilla SGD uses the same learning rate for all parameters — this is inefficient for ill-conditioned problems.

Modern optimizers adapt learning rates per-parameter and use momentum to accelerate convergence.

In [ ]:
# Visualize optimization trajectories on a 2D loss surface
import numpy as np
import matplotlib.pyplot as plt

# Loss surface: f(x,y) = 0.5*x^2 + 10*y^2 (steep in y, flat in x)
def loss_surface(x, y):
    return 0.5 * x**2 + 10 * y**2

def grad_surface(x, y):
    return np.array([x, 20 * y])

# Optimizer implementations
def sgd(start, lr=0.1, steps=50):
    path = [start]
    p = np.array(start, dtype=float)
    for _ in range(steps):
        g = grad_surface(p[0], p[1])
        p -= lr * g
        path.append(p.copy())
    return np.array(path)

def momentum_sgd(start, lr=0.1, mu=0.9, steps=50):
    path = [start]
    p = np.array(start, dtype=float)
    v = np.zeros_like(p)
    for _ in range(steps):
        g = grad_surface(p[0], p[1])
        v = mu * v - lr * g
        p += v
        path.append(p.copy())
    return np.array(path)

def adam(start, lr=0.1, beta1=0.9, beta2=0.999, eps=1e-8, steps=50):
    path = [start]
    p = np.array(start, dtype=float)
    m = np.zeros_like(p)
    v = np.zeros_like(p)
    for t in range(1, steps + 1):
        g = grad_surface(p[0], p[1])
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * g**2
        m_hat = m / (1 - beta1**t)
        v_hat = v / (1 - beta2**t)
        p -= lr * m_hat / (np.sqrt(v_hat) + eps)
        path.append(p.copy())
    return np.array(path)

start = (-4, 0.8)
paths = {
    'SGD': sgd(start, 0.1),
    'Momentum': momentum_sgd(start, 0.1),
    'Adam': adam(start, 0.1)
}

# Plot
x = np.linspace(-5, 5, 100)
y = np.linspace(-1.5, 1.5, 100)
X, Y = np.meshgrid(x, y)
Z = loss_surface(X, Y)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, path) in zip(axes, paths.items()):
    ax.contour(X, Y, Z, levels=20, cmap='viridis')
    ax.plot(path[:, 0], path[:, 1], 'r.-', markersize=4)
    ax.plot(start[0], start[1], 'ko', markersize=8)
    ax.set_title(name)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
plt.tight_layout()
plt.show()

## 2. Optimizer Details

| Optimizer | Update Rule | Key Features | Best For |
|-----------|-------------|--------------|----------|
| **SGD** | $\theta_{t+1} = \theta_t - \eta g_t$ | Simple, reliable | Small datasets, fine-tuning |
| **SGD+Momentum** | $v_{t+1} = \mu v_t + \eta g_t; \theta_{t+1} = \theta_t - v_{t+1}$ | Accelerates, dampens oscillations | General purpose |
| **SGD+Nesterov** | Look-ahead gradient | Faster convergence | When momentum works well |
| **AdaGrad** | $\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{G_t + \epsilon}} g_t$ | Per-param LR, adaptive | Sparse data (NLP) |
| **RMSProp** | $E[g^2]_t = \beta E[g^2]_{t-1} + (1-\beta) g_t^2$ | Fixes AdaGrad LR decay | Non-stationary objectives |
| **Adam** | Combines momentum + RMSProp | Fast, robust, default choice | **Most problems** |
| **AdamW** | Adam + decoupled weight decay | Better generalization | Transformers, large models |

In [ ]:
# Compare all optimizers on MNIST
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.reshape(-1, 784).astype('float32') / 255.0
x_test = x_test.reshape(-1, 784).astype('float32') / 255.0
y_train_cat = keras.utils.to_categorical(y_train)
y_test_cat = keras.utils.to_categorical(y_test)

def build_model():
    return keras.Sequential([
        layers.Dense(128, activation='relu', input_shape=(784,)),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])

optimizers = {
    'SGD': keras.optimizers.SGD(learning_rate=0.01),
    'SGD+Momentum': keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    'RMSProp': keras.optimizers.RMSprop(learning_rate=0.001),
    'Adam': keras.optimizers.Adam(learning_rate=0.001),
    'AdamW': keras.optimizers.AdamW(learning_rate=0.001)
}

opt_histories = {}
for name, opt in optimizers.items():
    model = build_model()
    model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])
    h = model.fit(x_train, y_train_cat, epochs=10, validation_split=0.2, verbose=0)
    test_acc = model.evaluate(x_test, y_test_cat, verbose=0)[1]
    opt_histories[name] = (h, test_acc)
    print(f"{name:20s}: Test acc = {test_acc:.4f}")

In [ ]:
# Plot optimizer comparison
plt.figure(figsize=(12, 4))
for name, (h, acc) in opt_histories.items():
    plt.plot(h.history['val_loss'], label=f'{name} ({acc:.3f})')
plt.title('Optimizer Comparison on MNIST (Validation Loss)')
plt.xlabel('Epoch')
plt.ylabel('Val Loss')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 3. Learning Rate Schedules in Keras

Keras provides built-in learning rate schedules:

In [ ]:
# Learning rate schedules comparison
initial_lr = 0.001
epochs = 30

# 1. ExponentialDecay
lr_exp = keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=initial_lr,
    decay_steps=500,
    decay_rate=0.9
)

# 2. CosineDecay
lr_cos = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=initial_lr,
    decay_steps=epochs * 600  # total steps
)

# 3. ReduceLROnPlateau (callback, not schedule)
reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6
)

print("Learning rate schedules configured:")
print(f"  ExponentialDecay: {lr_exp(0).numpy():.6f} (start)")
print(f"  CosineDecay: {lr_cos(0).numpy():.6f} (start)")

In [ ]:
# Train with ReduceLROnPlateau
model_rl = build_model()
model_rl.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

hist_rl = model_rl.fit(
    x_train, y_train_cat,
    epochs=20,
    validation_split=0.2,
    callbacks=[reduce_lr],
    verbose=1
)

print(f"Test accuracy with ReduceLROnPlateau: "
      f"{model_rl.evaluate(x_test, y_test_cat, verbose=0)[1]:.4f}")

## 4. Custom Learning Rate Schedule

You can implement custom schedules as functions or classes.

In [ ]:
# Custom learning rate schedule: cyclical LR
def cyclical_lr(step, base_lr=0.0001, max_lr=0.001, step_size=500):
    cycle = np.floor(1 + step / (2 * step_size))
    x = np.abs(step / step_size - 2 * cycle + 1)
    return base_lr + (max_lr - base_lr) * max(0, 1 - x)

# Plot
steps = np.arange(2000)
lrs = [cyclical_lr(s) for s in steps]

plt.figure(figsize=(10, 3))
plt.plot(steps, lrs)
plt.title('Cyclical Learning Rate')
plt.xlabel('Step')
plt.ylabel('Learning Rate')
plt.grid(alpha=0.3)
plt.show()

# Use as custom schedule in Keras
class CyclicalLR(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base_lr=0.0001, max_lr=0.001, step_size=500):
        self.base_lr = base_lr
        self.max_lr = max_lr
        self.step_size = step_size

    def __call__(self, step):
        cycle = tf.floor(1.0 + tf.cast(step, tf.float32) / (2 * self.step_size))
        x = tf.abs(tf.cast(step, tf.float32) / self.step_size - 2 * cycle + 1)
        return self.base_lr + (self.max_lr - self.base_lr) * tf.maximum(0.0, 1.0 - x)

    def get_config(self):
        return {'base_lr': self.base_lr, 'max_lr': self.max_lr, 'step_size': self.step_size}

## 5. Choosing the Right Optimizer

**General guidelines:**

| Scenario | Recommended Optimizer |
|----------|----------------------|
| **Default choice** | Adam (LR=0.001) |
| **Computer Vision** | SGD+Momentum or AdamW |
| **NLP / Transformers** | AdamW with warmup |
| **Small dataset** | SGD+Momentum |
| **Sparse features** | AdaGrad |
| **GANs** | Adam (often with lower LR) |
| **Reinforcement Learning** | Adam |
| **Fine-tuning** | SGD (low LR) |

**Pro tip:** Always start with Adam. If you need the last 1-2% of performance, switch to SGD+Momentum with a well-tuned LR schedule.

## 6. Interview Problems

### Problem 1: Compare Optimizer Performance on MNIST

Train the same architecture with SGD, SGD+Momentum, Adam, and AdamW. Plot training curves and report final test accuracy.

### Problem 2: Implement Custom Learning Rate Schedule

Implement a warmup + cosine decay schedule. Train for 10 epochs with warmup (first 2 epochs) followed by cosine decay.

### Problem 3: Diagnose Optimizer Issues

Given loss curves that oscillate wildly, plateau early, or explode — identify which optimizer/hyperparameter is the likely cause.

In [ ]:
# Solution: Problem 2 — Warmup + Cosine Decay
class WarmupCosineDecay(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, warmup_steps=500, total_steps=3000, max_lr=0.001, min_lr=0.0):
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.max_lr = max_lr
        self.min_lr = min_lr

    def __call__(self, step):
        step_f = tf.cast(step, tf.float32)
        warmup = self.warmup_steps
        total = self.total_steps

        # Linear warmup
        warmup_lr = self.max_lr * (step_f / warmup)

        # Cosine decay after warmup
        cosine_lr = self.min_lr + 0.5 * (self.max_lr - self.min_lr) * (
            1 + tf.cos(np.pi * (step_f - warmup) / (total - warmup))
        )

        return tf.where(step_f < warmup, warmup_lr, cosine_lr)

    def get_config(self):
        return {'warmup_steps': self.warmup_steps, 'total_steps': self.total_steps,
                'max_lr': self.max_lr, 'min_lr': self.min_lr}

# Test the schedule
scheduler = WarmupCosineDecay(warmup_steps=500, total_steps=3000)
lrs = [scheduler(s).numpy() for s in range(3000)]

plt.figure(figsize=(10, 3))
plt.plot(lrs)
plt.title('Warmup + Cosine Decay')
plt.xlabel('Step')
plt.ylabel('Learning Rate')
plt.grid(alpha=0.3)
plt.show()

## Summary

- **SGD** is simple but slow — needs momentum for practical use
- **AdaGrad** adapts LR per-parameter but LR monotonically decreases
- **RMSProp** fixes AdaGrad's aggressive decay with moving average
- **Adam** combines momentum + RMSProp — the default optimizer
- **AdamW** decouples weight decay for better generalization
- **Learning rate schedules** (exponential, cosine, ReduceLROnPlateau) improve convergence
- **Cyclical LR** oscillates between bounds for faster exploration
- Start with Adam; switch to SGD+Momentum for final tuning